<a href="https://colab.research.google.com/github/melissa-04/tubitak-2209a-spatial-stemness-emt/blob/main/09_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q scanpy statsmodels "pandas==2.3.3"

from google.colab import drive
drive.mount('/content/drive')

import os, sys
import scanpy as sc, pandas as pd, numpy as np
from scipy.stats import spearmanr, wilcoxon, rankdata
sc.settings.verbosity = 0

BASE    = '/content/drive/MyDrive/Tubitak-2209a'
RESULTS = f'{BASE}/04_results'
FIGURES = f'{BASE}/05_figures'
os.makedirs(RESULTS, exist_ok=True)

scores = pd.read_csv(f'{BASE}/03_spatial_analysis/spatial_all_scores.csv')
emt    = pd.read_csv(f'{BASE}/03_spatial_analysis/spatial_emt_scores.csv')
sig_c2l = pd.read_csv(f'{BASE}/03_spatial_analysis/c2l_reference_signatures.csv', index_col=0)
regions = pd.read_csv(f'{BASE}/01_processed_data/combined_with_regions.csv')
regions['SpotID'] = regions.patientid.astype(str) + '_' + regions.barcode.astype(str)

df = scores.merge(regions[['SpotID', 'array_row', 'array_col', 'nCount_RNA', 'nFeature_RNA']],
                  on='SpotID', how='left')
for c in ['EMT_tan_epi', 'EMT_tan_mes']:
    if c in emt.columns:
        df[c] = emt.set_index('SpotID').reindex(df.SpotID)[c].values

print(f'spots         : {df.shape[0]} ({df.spatial_stemness.notna().sum()} with stemness)')
print(f'c2l signature : {sig_c2l.shape[0]} genes x {sig_c2l.shape[1]} cell types')
print(f'Tan sub-scores present: {[c for c in ["EMT_tan_epi","EMT_tan_mes"] if c in df.columns]}')

sig_c2l.columns = [c.replace('meanscell_abundance_w_sf_', '')
                    .replace('means_per_cluster_mu_fg_', '') for c in sig_c2l.columns]
print(f'cell types    : {list(sig_c2l.columns)}')

PATIENTS = sorted(df.patientid.unique())
SUBTYPE  = df.groupby('patientid', observed=True).subtype.first().to_dict()
EMT_SIGS = ['pEMT_puram', 'EMT_hallmark', 'EMT_tan']
d = df[df.spatial_stemness.notna()].copy()

def rho_by_patient(frame, x, y, min_n=50):
    out = []
    for p, s in frame.groupby('patientid', observed=True):
        s = s[[x, y]].dropna()
        if len(s) < min_n:
            continue
        out.append(spearmanr(s[x], s[y])[0])
    return [r for r in out if not np.isnan(r)]

def summarise(rs, label):
    try:
        wp = wilcoxon(rs)[1]
    except Exception:
        wp = np.nan
    return {'analysis': label, 'n_patients': len(rs),
            'median_rho': round(np.median(rs), 4),
            'n_positive': f'{sum(r > 0 for r in rs)}/{len(rs)}',
            'wilcoxon_p': round(wp, 4)}

print('\n=== REFERENCE: the main finding as reported so far ===')
REF = []
for sig in EMT_SIGS:
    rs = rho_by_patient(d, 'spatial_stemness', sig)
    REF.append(summarise(rs, f'reference: {sig}'))
ref = pd.DataFrame(REF)
print(ref.to_string(index=False))
print('\nEvery sensitivity test below is compared against these numbers.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas

In [ ]:
!pip install gseapy

vis = sc.read_h5ad(f'{BASE}/03_spatial_analysis/visium_6patients_raw.h5ad')
sc.pp.normalize_total(vis, target_sum=1e4); sc.pp.log1p(vis)
vis.obs['SpotID'] = vis.obs_names
print(f'Visium normalized: {vis.shape}')

import gseapy
PURAM = ["SERPINE1","TGFBI","MMP10","LAMC2","P4HA2","PDPN","ITGA5","LAMA3","CDH13","TNC",
"MMP2","EMP3","INHBA","LAMB3","VIM","SEMA3C","CAVIN3","ANXA5","DHRS7","ITGB1","ACTN1",
"ACKR3","ITGB6","IGFBP7","THBS1","PTHLH","TNFRSF6B","PDLIM7","CAV1","DKK3","COL17A1",
"LTBP1","COL5A2","COL1A1","FHL2","TIMP3","PLAU","LGALS1","PSMD2","CD63","HERPUD1","TPM1",
"SLC39A14","C1S","MMP1","EXT2","COL4A2","PRSS23","SLC7A8","SLC31A2","ARPC1B","APP","MFAP2",
"MPZL1","GSDME","MT2A","MAGED2","ITGA6","FSTL1","TNFRSF12A","IL32","COPB2","PTK7","OCIAD2",
"TAX1BP3","SEC13","SERPINH1","TPM4","MYH9","ANXA8L1","PLOD2","GALNT2","P3H2","MAGED1",
"SLC38A5","FSTL3","CD99","F3","PSAP","NMRK1","FKBP9","DSG2","ECM1","HTRA1","SERINC1","CALU",
"TPST1","PLOD3","IGFBP3","FRMD6","CXCL14","SERPINE2","RABAC1","TMED9","NAGK","BMP1","ESYT1",
"STON2","TAGLN","GJA1"]
HALLMARK = gseapy.get_library("MSigDB_Hallmark_2020")["Epithelial Mesenchymal Transition"]
RAW_SIGS = {'pEMT_puram': PURAM, 'EMT_hallmark': HALLMARK}

EPI_TYPES = ['Cancer Epithelial', 'Normal Epithelial']
STROMAL   = ['CAFs', 'PVL', 'Endothelial']

def epithelial_ratio(genes):
    g = [x for x in genes if x in sig_c2l.index]
    S = sig_c2l.loc[g]
    return (S[EPI_TYPES].max(axis=1) / (S[STROMAL].max(axis=1) + 1e-9)), S

FILTERS = {'none': None, 'lenient': 0.5, 'moderate': 1.0, 'strict': 2.0}

print('\n=== GENE COMPOSITION OF EACH SIGNATURE ===')
kept_sets = {}
for name, genes in RAW_SIGS.items():
    ratio, S = epithelial_ratio(genes)
    dom = S.idxmax(axis=1)
    print(f'\n{name}: {len(ratio)} genes in reference matrix')
    print('  dominant cell type: ' +
          ', '.join(f'{k}={v}' for k, v in dom.value_counts().head(5).items()))
    for fname, thr in FILTERS.items():
        kept = list(ratio.index) if thr is None else list(ratio[ratio >= thr].index)
        kept = [g for g in kept if g in vis.var_names]
        kept_sets[(name, fname)] = kept
        print(f'    filter {fname:9s} (epi/stromal >= {thr}): {len(kept):3d} genes kept')

print('\n=== SCORING AND RE-TESTING ===')
rng = np.random.default_rng(42)
rows = []
for name in RAW_SIGS:
    for fname in FILTERS:
        genes = kept_sets[(name, fname)]
        if len(genes) < 10:
            print(f'  {name} / {fname}: only {len(genes)} genes, skipped')
            continue
        col = f'{name}__{fname}'
        sc.tl.score_genes(vis, gene_list=genes, score_name=col,
                          ctrl_size=50, random_state=42, use_raw=False)
        d[col] = vis.obs.set_index('SpotID').reindex(d.SpotID)[col].values
        rs = rho_by_patient(d, 'spatial_stemness', col)
        r = summarise(rs, f'{name} / filter={fname} (n_genes={len(genes)})')
        r['n_genes'] = len(genes); r['signature'] = name; r['filter'] = fname
        rows.append(r)

        if fname != 'none':
            n_keep = len(genes)
            all_g = [g for g in RAW_SIGS[name] if g in vis.var_names]
            rand_meds = []
            for rep in range(20):
                rsub = list(rng.choice(all_g, size=n_keep, replace=False))
                rcol = f'_rand_{rep}'
                sc.tl.score_genes(vis, gene_list=rsub, score_name=rcol,
                                  ctrl_size=50, random_state=42, use_raw=False)
                d[rcol] = vis.obs.set_index('SpotID').reindex(d.SpotID)[rcol].values
                rand_meds.append(np.median(rho_by_patient(d, 'spatial_stemness', rcol)))
                d.drop(columns=rcol, inplace=True)
            rows[-1]['random_ctrl_median'] = round(np.median(rand_meds), 4)
            rows[-1]['random_ctrl_sd'] = round(np.std(rand_meds), 4)

refined = pd.DataFrame(rows)
print()
print(refined[['signature', 'filter', 'n_genes', 'median_rho', 'n_positive',
               'wilcoxon_p', 'random_ctrl_median', 'random_ctrl_sd']].to_string(index=False))
print('\n  random_ctrl = same number of genes removed at random (20 repeats)')
print('  if filtered < random_ctrl, the drop is due to CAF genes, not gene count')

refined.to_csv(f'{RESULTS}/refined_signature.csv', index=False)
print(f'\nSaved -> {RESULTS}/refined_signature.csv')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.7/689.7 kB 31.3 MB/s eta 0:00:00
Visium normalized: (15611, 28925)

=== GENE COMPOSITION OF EACH SIGNATURE ===

pEMT_puram: 97 genes in reference matrix
  dominant cell type: CAFs=37, Normal Epithelial=18, Cancer Epithelial=17, Endothelial=12, Myeloid=6
    filter none      (epi/stromal >= None):  97 genes kept
    filter lenient   (epi/stromal >= 0.5):  58 genes kept
    filter moderate  (epi/stromal >= 1.0):  38 genes kept
    filter strict    (epi/stromal >= 2.0):  18 genes kept

EMT_hallmark: 196 genes in reference matrix
  dominant cell type: CAFs=100, Normal Epithelial=26, Endothelial=20, Myeloid=17, Cancer Epithelial=17
    filter none      (epi/stromal >= None): 196 genes kept
    filter lenient   (epi/stromal >= 0.5):  72 genes kept
    filter moderate  (epi/stromal >= 1.0):  49 genes kept
    filter strict    (epi/stromal >= 2.0):  27 genes kept

=== SCORING AND RE-TESTING ===

   signature   filter  n_genes  median_rho n_positi

In [ ]:
print('=== (a) DOES THE REFINED SCORE STILL MEASURE THE SAME THING? ===\n')
for name in RAW_SIGS:
    print(f'{name}:')
    for f in ['lenient', 'moderate', 'strict']:
        col = f'{name}__{f}'
        if col not in d.columns:
            continue
        r_orig = spearmanr(d[col], d[name])[0]
        r_caf  = spearmanr(d[col], d['CAFs'])[0]
        r_epi  = spearmanr(d[col], d['Cancer Epithelial'])[0]
        print(f'  {f:9s} vs original={r_orig:+.3f} | vs CAF fraction={r_caf:+.3f} '
              f'| vs CancerEpi fraction={r_epi:+.3f}')
    print(f'  {"original":9s} vs original=+1.000 | vs CAF fraction='
          f'{spearmanr(d[name], d["CAFs"])[0]:+.3f} | vs CancerEpi fraction='
          f'{spearmanr(d[name], d["Cancer Epithelial"])[0]:+.3f}\n')

print('=== (b) WHICH GENES SURVIVE THE STRICT FILTER? ===\n')
for name in RAW_SIGS:
    genes = kept_sets[(name, 'strict')]
    ratio, S = epithelial_ratio(RAW_SIGS[name])
    sub = ratio.loc[[g for g in genes if g in ratio.index]].sort_values(ascending=False)
    print(f'{name} ({len(genes)} genes, epi/stromal ratio):')
    print('  ' + ', '.join(f'{g}({v:.1f})' for g, v in sub.items()))
    print()

print('=== (c) DOES THE REFINED SCORE STILL BEHAVE LIKE EMT? ===')
print('    checking against CDH1 (epithelial, should be NEGATIVE if EMT is measured)\n')
cdh1 = np.asarray(vis[:, 'CDH1'].X.todense()).ravel()
d['_CDH1'] = pd.Series(cdh1, index=vis.obs_names).reindex(d.SpotID).values
for name in RAW_SIGS:
    rs_o = rho_by_patient(d, '_CDH1', name)
    print(f'{name:14s} original : median rho vs CDH1 = {np.median(rs_o):+.3f} '
          f'({sum(r<0 for r in rs_o)}/{len(rs_o)} negative)')
    for f in ['moderate', 'strict']:
        col = f'{name}__{f}'
        if col in d.columns:
            rs_f = rho_by_patient(d, '_CDH1', col)
            print(f'{"":14s} {f:9s}: median rho vs CDH1 = {np.median(rs_f):+.3f} '
                  f'({sum(r<0 for r in rs_f)}/{len(rs_f)} negative)')
    print()

=== (a) DOES THE REFINED SCORE STILL MEASURE THE SAME THING? ===

pEMT_puram:
  lenient   vs original=+0.776 | vs CAF fraction=+0.280 | vs CancerEpi fraction=-0.287
  moderate  vs original=+0.633 | vs CAF fraction=+0.220 | vs CancerEpi fraction=-0.167
  strict    vs original=+0.273 | vs CAF fraction=+0.132 | vs CancerEpi fraction=+0.014
  original  vs original=+1.000 | vs CAF fraction=+0.698 | vs CancerEpi fraction=-0.514

EMT_hallmark:
  lenient   vs original=+0.349 | vs CAF fraction=+0.069 | vs CancerEpi fraction=-0.236
  moderate  vs original=+0.247 | vs CAF fraction=+0.000 | vs CancerEpi fraction=-0.112
  strict    vs original=+0.028 | vs CAF fraction=-0.146 | vs CancerEpi fraction=+0.014
  original  vs original=+1.000 | vs CAF fraction=+0.899 | vs CancerEpi fraction=-0.581

=== (b) WHICH GENES SURVIVE THE STRICT FILTER? ===

pEMT_puram (18 genes, epi/stromal ratio):
  ITGB6(151.9), LAMC2(65.5), ANXA8L1(21.7), MMP10(11.0), LAMB3(10.7), DSG2(5.5), COL17A1(4.4), PTHLH(3.6), PSMD2(3.5

In [ ]:
assign = pd.read_csv(f'{BASE}/03_spatial_analysis/cytospace_assignments_all.csv')
cell_sc = pd.read_csv(f'{RESULTS}/cell_level_scores.csv', index_col=0)
print(f'assignments : {len(assign)} rows, cols={list(assign.columns)[:8]}')
print(f'cell scores : {len(cell_sc)} cells')

EPI_CT = ['Cancer Epithelial', 'Normal Epithelial']
epi_as = assign[assign.CellType.isin(EPI_CT)].copy()
print(f'\nepithelial assignments: {len(epi_as)}')

epi_as = epi_as.merge(cell_sc[['pEMT_puram', 'EMT_hallmark', 'CT2']],
                      left_on='OriginalCID', right_index=True, how='left')
matched = epi_as.pEMT_puram.notna().sum()
print(f'  matched to cell scores: {matched}/{len(epi_as)} ({100*matched/len(epi_as):.1f}%)')

if 'patientid' not in epi_as.columns:
    epi_as['patientid'] = epi_as.SpotID.str.split('_').str[0]
epi_as['SpotKey'] = epi_as.SpotID if epi_as.SpotID.iloc[0].count('_') > 0 else \
                    epi_as.patientid + '_' + epi_as.SpotID

carried = epi_as.groupby('SpotKey').agg(
    pEMT_carried=('pEMT_puram', 'mean'),
    Hallmark_carried=('EMT_hallmark', 'mean'),
    CT2_carried=('CT2', 'mean'),
    n_epi=('OriginalCID', 'size')).reset_index()
print(f'\nspots with carried scores: {len(carried)}')

d2 = d.merge(carried, left_on='SpotID', right_on='SpotKey', how='left')
ok = d2.pEMT_carried.notna().sum()
print(f'merged onto stemness spots: {ok}/{len(d2)}')

if ok < 1000:
    print('\n!! SpotID format mismatch - inspect these:')
    print('  assignment SpotKey sample:', carried.SpotKey.head(3).tolist())
    print('  score table SpotID sample:', d.SpotID.head(3).tolist())
else:
    print('\n--- CONSISTENCY CHECK: carried CT2 vs stored spatial_stemness ---')
    r_chk = spearmanr(d2.CT2_carried.dropna(),
                      d2.loc[d2.CT2_carried.notna(), 'spatial_stemness'])[0]
    print(f'  rho = {r_chk:+.4f}  (should be ~1.0 if the pipeline is reproduced correctly)')

    print('\n=== STEMNESS x EMT: SPOT EXPRESSION vs CARRIED EPITHELIAL SCORE ===\n')
    rows = []
    pairs = [('spot expression', 'pEMT_puram'), ('carried epithelial', 'pEMT_carried'),
             ('spot expression', 'EMT_hallmark'), ('carried epithelial', 'Hallmark_carried')]
    for lbl, col in pairs:
        rs = rho_by_patient(d2, 'spatial_stemness', col)
        r = summarise(rs, f'{col} [{lbl}]')
        rows.append(r)
        print(f'  {col:20s} [{lbl:18s}] median rho={r["median_rho"]:+.4f}  '
              f'{r["n_positive"]}  p={r["wilcoxon_p"]}')

    print('\n--- how much of the spot signal is explained by the carried score? ---')
    for orig, carr in [('pEMT_puram', 'pEMT_carried'), ('EMT_hallmark', 'Hallmark_carried')]:
        r_oc = spearmanr(d2[orig].dropna(),
                         d2.loc[d2[orig].notna(), carr].fillna(d2[carr].mean()))[0]
        print(f'  {orig:14s} vs {carr:18s}: rho={r_oc:+.3f}')

    print('\n--- reference points from step 3 ---')
    print('  cell level (no mixing)     : +0.037')
    print('  pseudo-spots (random 8)    : +0.076')
    print('  real spots (expression)    : +0.196')

    pd.DataFrame(rows).to_csv(f'{RESULTS}/carried_emt_comparison.csv', index=False)
    print(f'\nSaved -> {RESULTS}/carried_emt_comparison.csv')

assignments : 110560 rows, cols=['UniqueCID', 'OriginalCID', 'CellType', 'SpotID', 'row', 'col', 'patientid']
cell scores : 28844 cells

epithelial assignments: 63976
  matched to cell scores: 63976/63976 (100.0%)

spots with carried scores: 10263
merged onto stemness spots: 10263/10263

--- CONSISTENCY CHECK: carried CT2 vs stored spatial_stemness ---
  rho = +1.0000  (should be ~1.0 if the pipeline is reproduced correctly)

=== STEMNESS x EMT: SPOT EXPRESSION vs CARRIED EPITHELIAL SCORE ===

  pEMT_puram           [spot expression   ] median rho=+0.1962  6/6  p=0.0312
  pEMT_carried         [carried epithelial] median rho=+0.5130  6/6  p=0.0312
  EMT_hallmark         [spot expression   ] median rho=+0.2619  6/6  p=0.0312
  Hallmark_carried     [carried epithelial] median rho=+0.5962  6/6  p=0.0312

--- how much of the spot signal is explained by the carried score? ---
  pEMT_puram     vs pEMT_carried      : rho=+0.353
  EMT_hallmark   vs Hallmark_carried  : rho=+0.400

--- reference 

In [ ]:
rng = np.random.default_rng(42)

print('=== HOW MUCH OF THE CARRIED CORRELATION COMES FROM THE ASSIGNMENT ITSELF? ===')
print('    shuffling which cells go to which spot, within patient, 100 repeats\n')

ea = epi_as[epi_as.pEMT_puram.notna()].copy()
ea['SpotKey'] = ea.SpotKey.astype(str)

for emt_col, carr_name in [('pEMT_puram', 'pEMT'), ('EMT_hallmark', 'Hallmark')]:
    obs_rs = rho_by_patient(d2, 'spatial_stemness',
                            'pEMT_carried' if carr_name == 'pEMT' else 'Hallmark_carried')
    null_meds = []
    for rep in range(100):
        rr = np.random.default_rng(500 + rep)
        parts = []
        for p, sub in ea.groupby('patientid', observed=True):
            sub = sub.copy()
            sub['SpotKey'] = rr.permutation(sub.SpotKey.values)
            parts.append(sub)
        sh = pd.concat(parts)
        agg = sh.groupby('SpotKey').agg(e=(emt_col, 'mean'), c=('CT2', 'mean')).reset_index()
        agg['patientid'] = agg.SpotKey.str.split('_').str[0]
        rs = []
        for p, s in agg.groupby('patientid', observed=True):
            if len(s) >= 50:
                rs.append(spearmanr(s.c, s.e)[0])
        null_meds.append(np.median(rs))
    null_meds = np.array(null_meds)
    print(f'  {carr_name}')
    print(f'    observed (real assignment)    : {np.median(obs_rs):+.3f}')
    print(f'    shuffled assignment (null)    : {null_meds.mean():+.3f} +/- {null_meds.std():.3f}')
    print(f'    excess over null              : {np.median(obs_rs) - null_meds.mean():+.3f}')
    print(f'    empirical p                   : '
          f'{(np.sum(null_meds >= np.median(obs_rs)) + 1) / (len(null_meds) + 1):.4f}\n')

print('Interpretation:')
print('  null captures: averaging arithmetic + patient composition')
print('  excess captures: what the expression-similarity assignment adds')

=== HOW MUCH OF THE CARRIED CORRELATION COMES FROM THE ASSIGNMENT ITSELF? ===
    shuffling which cells go to which spot, within patient, 100 repeats

  pEMT
    observed (real assignment)    : +0.513
    shuffled assignment (null)    : +0.463 +/- 0.012
    excess over null              : +0.050
    empirical p                   : 0.0099

  Hallmark
    observed (real assignment)    : +0.596
    shuffled assignment (null)    : +0.520 +/- 0.011
    excess over null              : +0.077
    empirical p                   : 0.0099

Interpretation:
  null captures: averaging arithmetic + patient composition
  excess captures: what the expression-similarity assignment adds


In [ ]:
!pip install -q decoupler

import decoupler as dc
print(f'decoupler {dc.__version__}')

net = pd.concat([
    pd.DataFrame({'source': 'pEMT_puram',
                  'target': [g for g in PURAM if g in vis.var_names], 'weight': 1.0}),
    pd.DataFrame({'source': 'EMT_hallmark',
                  'target': [g for g in HALLMARK if g in vis.var_names], 'weight': 1.0}),
])
print(f'\nnetwork: {net.source.nunique()} signatures, '
      f'{net.groupby("source").size().to_dict()}')

vis_df = pd.DataFrame(vis.X.toarray() if hasattr(vis.X, 'toarray') else vis.X,
                      index=vis.obs_names, columns=vis.var_names)
print(f'expression matrix ready: {vis_df.shape}')

METHODS = {}

try:
    res = dc.mt.ulm(data=vis_df, net=net, verbose=False)
    METHODS['ulm'] = res[0] if isinstance(res, tuple) else res
    print('  ulm     : ok')
except Exception as e:
    print(f'  ulm     : failed ({str(e)[:60]})')

try:
    import scanpy as sc_
    ad_tmp = sc.AnnData(X=vis_df.values)
    ad_tmp.obs_names = vis_df.index; ad_tmp.var_names = vis_df.columns
    dc.mt.aucell(ad_tmp, net=net, verbose=False)
    key = [k for k in ad_tmp.obsm.keys() if 'aucell' in k.lower()]
    METHODS['aucell'] = ad_tmp.obsm[key[0]] if key else None
    print('  aucell  : ok')
except Exception as e:
    print(f'  aucell  : failed ({str(e)[:60]})')

try:
    ad_tmp2 = sc.AnnData(X=vis_df.values)
    ad_tmp2.obs_names = vis_df.index; ad_tmp2.var_names = vis_df.columns
    dc.mt.udt(ad_tmp2, net=net, verbose=False) if hasattr(dc.mt, 'udt') else None
    print('  (udt skipped)')
except Exception:
    pass

print('\navailable decoupler methods:', [m for m in dir(dc.mt) if not m.startswith('_')][:15])

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.4/147.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.9 MB/s eta 0:00:00
decoupler 2.2.0

network: 2 signatures, {'EMT_hallmark': 200, 'pEMT_puram': 100}
expression matrix ready: (15611, 28925)
  ulm     : ok
  aucell  : ok
  (udt skipped)

available decoupler methods: ['aucell', 'consensus', 'decouple', 'gsea', 'gsva', 'mdt', 'mlm', 'ora', 'query_set', 'show', 'udt', 'ulm', 'viper', 'waggr', 'zscore']


In [2]:
subsets = {
    'all spots with stemness'      : d,
    'tumor spots only'             : d[d.region_class == 'tumor'],
    'analyze == True'              : d[d.analyze == True],
    'tumor & analyze'              : d[(d.region_class == 'tumor') & (d.analyze == True)],
    'CancerEpi >= 0.40'            : d[d['Cancer Epithelial'] >= 0.40],
    'CancerEpi >= 0.50'            : d[d['Cancer Epithelial'] >= 0.50],
    'CancerEpi >= 0.60'            : d[d['Cancer Epithelial'] >= 0.60],
    'n_epi_cells >= 3'             : d[d.n_epi_cells >= 3],
    'n_epi_cells >= 5'             : d[d.n_epi_cells >= 5],
    'tumor & CancerEpi >= 0.50'    : d[(d.region_class == 'tumor') & (d['Cancer Epithelial'] >= 0.50)],
}

rows = []
for label, sub in subsets.items():
    for sig in EMT_SIGS:
        rs = rho_by_patient(sub, 'spatial_stemness', sig)
        if len(rs) < 3:
            continue
        r = summarise(rs, label)
        r['signature'] = sig
        r['n_spots'] = len(sub)
        rows.append(r)

sens = pd.DataFrame(rows)
print('=== SUBSET SENSITIVITY ===\n')
for sig in EMT_SIGS:
    print(f'{sig}:')
    v = sens[sens.signature == sig][['analysis', 'n_spots', 'n_patients',
                                      'median_rho', 'n_positive', 'wilcoxon_p']]
    print(v.to_string(index=False))
    print()

print('=== BY SUBTYPE ===\n')
sub_rows = []
for st in ['TNBC', 'ER']:
    pats = [p for p in PATIENTS if SUBTYPE[p] == st]
    sub = d[d.patientid.isin(pats)]
    for sig in EMT_SIGS:
        rs = rho_by_patient(sub, 'spatial_stemness', sig)
        sub_rows.append({'subtype': st, 'n_patients': len(rs), 'signature': sig,
                         'median_rho': round(np.median(rs), 4),
                         'n_positive': f'{sum(r > 0 for r in rs)}/{len(rs)}',
                         'individual': ', '.join(f'{p}={r:+.3f}' for p, r in zip(pats, rs))})
print(pd.DataFrame(sub_rows).to_string(index=False))
print('\n  ER+ has only 2 patients - descriptive only, no inference')

print('\n=== LEAVE-ONE-PATIENT-OUT ===\n')
loo_rows = []
for sig in EMT_SIGS:
    full = np.median(rho_by_patient(d, 'spatial_stemness', sig))
    for p in PATIENTS:
        rs = rho_by_patient(d[d.patientid != p], 'spatial_stemness', sig)
        loo_rows.append({'signature': sig, 'excluded': p, 'subtype': SUBTYPE[p],
                         'median_rho': round(np.median(rs), 4),
                         'n_positive': f'{sum(r > 0 for r in rs)}/{len(rs)}',
                         'shift_from_full': round(np.median(rs) - full, 4)})
loo = pd.DataFrame(loo_rows)
for sig in EMT_SIGS:
    print(f'{sig} (full cohort median = {np.median(rho_by_patient(d, "spatial_stemness", sig)):+.4f}):')
    print(loo[loo.signature == sig].drop(columns='signature').to_string(index=False))
    print()

sens.to_csv(f'{RESULTS}/sensitivity_subsets.csv', index=False)
pd.DataFrame(sub_rows).to_csv(f'{RESULTS}/sensitivity_subtype.csv', index=False)
loo.to_csv(f'{RESULTS}/sensitivity_leave_one_out.csv', index=False)
print(f'Saved -> sensitivity_subsets.csv, sensitivity_subtype.csv, sensitivity_leave_one_out.csv')

=== SUBSET SENSITIVITY ===

pEMT_puram:
                 analysis  n_spots  n_patients  median_rho n_positive  wilcoxon_p
  all spots with stemness    10263           6      0.1962        6/6      0.0312
         tumor spots only     8583           6      0.2803        6/6      0.0312
          analyze == True    10263           6      0.1962        6/6      0.0312
          tumor & analyze     8583           6      0.2803        6/6      0.0312
        CancerEpi >= 0.40     8382           6      0.1055        5/6      0.1562
        CancerEpi >= 0.50     6310           6      0.0686        4/6      0.3125
        CancerEpi >= 0.60     4118           5      0.0245        3/5      0.4375
         n_epi_cells >= 3     8399           6      0.1603        5/6      0.0625
         n_epi_cells >= 5     6961           6      0.1348        5/6      0.0625
tumor & CancerEpi >= 0.50     5729           5      0.0246        3/5      0.6250

EMT_hallmark:
                 analysis  n_spots  n_patie

In [3]:
print('=== (a) FRONT DEFINITION: THRESHOLD SENSITIVITY ===\n')
dt = df[df.region_class == 'tumor'].copy()
variants = {}
for ring2_min in [1, 2, 3]:
    variants[f'front: >={ring2_min} nontumor in 2-ring'] = dt.n_nontumor_2ring >= ring2_min
for valid_min in [20, 24, 28]:
    variants[f'core: 0 in 3-ring & >={valid_min}/36 valid'] = \
        (dt.n_nontumor_3ring == 0) & (dt.n_valid_3ring >= valid_min)

print('Region counts under each rule variant:')
for lbl, mask in variants.items():
    print(f'  {lbl:44s} -> {mask.sum():5d} spots')

rows = []
for r2 in [1, 2, 3]:
    for vm in [20, 24, 28]:
        dt['_region'] = 'uncertain'
        dt.loc[dt.n_nontumor_2ring >= r2, '_region'] = 'front'
        core_m = (dt.n_nontumor_2ring < r2) & (dt.n_nontumor_3ring == 0) & (dt.n_valid_3ring >= vm)
        dt.loc[core_m, '_region'] = 'core'
        sub = dt[dt.spatial_stemness.notna()]
        ds_ = []
        for p in PATIENTS:
            s = sub[sub.patientid == p]
            f = s[s._region == 'front'].spatial_stemness.values
            c = s[s._region == 'core'].spatial_stemness.values
            if len(f) < 10 or len(c) < 10:
                continue
            sp = np.sqrt(((len(f)-1)*f.var(ddof=1) + (len(c)-1)*c.var(ddof=1)) / (len(f)+len(c)-2))
            ds_.append((f.mean() - c.mean()) / sp if sp > 0 else np.nan)
        ds_ = [x for x in ds_ if not np.isnan(x)]
        rows.append({'front_rule': f'>={r2} in 2ring', 'core_rule': f'>={vm}/36 valid',
                     'n_front': int((dt._region == 'front').sum()),
                     'n_core': int((dt._region == 'core').sum()),
                     'n_testable': len(ds_),
                     'median_d': round(np.median(ds_), 3) if ds_ else np.nan,
                     'front_higher': f'{sum(x > 0 for x in ds_)}/{len(ds_)}' if ds_ else '-'})

thr = pd.DataFrame(rows)
print('\nfront vs core stemness (Cohen\'s d) across 9 rule combinations:')
print(thr.to_string(index=False))
print('\n  if median_d stays small and inconsistent across all variants,')
print('  the null result is not an artefact of threshold choice')

print('\n\n=== (b) TAN SIGNATURE: WHICH ARM CARRIES THE SIGNAL? ===\n')
tan_rows = []
for col, desc in [('EMT_tan', 'combined (mes - epi)'),
                  ('EMT_tan_mes', 'mesenchymal arm only'),
                  ('EMT_tan_epi', 'epithelial arm only')]:
    rs = rho_by_patient(d, 'spatial_stemness', col)
    r = summarise(rs, f'{col}  [{desc}]')
    r['vs_CAF'] = round(spearmanr(d[col], d['CAFs'])[0], 3)
    r['vs_CancerEpi'] = round(spearmanr(d[col], d['Cancer Epithelial'])[0], 3)
    tan_rows.append(r)
tan = pd.DataFrame(tan_rows)
print(tan.to_string(index=False))

print('\nunder five-covariate control:')
COV = ['Cancer Epithelial', 'CAFs', 'nCount_RNA', 'n_epi_cells']
for col in ['EMT_tan', 'EMT_tan_mes', 'EMT_tan_epi']:
    ps = [partial_rho(d[d.patientid == p], 'spatial_stemness', col, COV) for p in PATIENTS]
    ps = [x for x in ps if not np.isnan(x)]
    print(f'  {col:14s} partial rho = {np.median(ps):+.4f}  '
          f'({sum(x > 0 for x in ps)}/{len(ps)} positive)')

thr.to_csv(f'{RESULTS}/sensitivity_thresholds.csv', index=False)
tan.to_csv(f'{RESULTS}/sensitivity_tan_arms.csv', index=False)
print(f'\nSaved -> sensitivity_thresholds.csv, sensitivity_tan_arms.csv')

=== (a) FRONT DEFINITION: THRESHOLD SENSITIVITY ===

Region counts under each rule variant:
  front: >=1 nontumor in 2-ring                ->  1732 spots
  front: >=2 nontumor in 2-ring                ->  1382 spots
  front: >=3 nontumor in 2-ring                ->  1091 spots
  core: 0 in 3-ring & >=20/36 valid            ->  8299 spots
  core: 0 in 3-ring & >=24/36 valid            ->  7783 spots
  core: 0 in 3-ring & >=28/36 valid            ->  7262 spots

front vs core stemness (Cohen's d) across 9 rule combinations:
  front_rule     core_rule  n_front  n_core  n_testable  median_d front_higher
>=1 in 2ring >=20/36 valid     1732    8299           4     0.229          3/4
>=1 in 2ring >=24/36 valid     1732    7783           4     0.236          3/4
>=1 in 2ring >=28/36 valid     1732    7262           4     0.239          3/4
>=2 in 2ring >=20/36 valid     1382    8299           4     0.216          3/4
>=2 in 2ring >=24/36 valid     1382    7783           4     0.222          3/

In [4]:
import statsmodels.api as sm

print('=== (a) QUINTILE PROFILE: mean stemness across EMT bins ===\n')
q_rows = []
for sig in EMT_SIGS + ['EMT_tan_epi']:
    profs = []
    for p in PATIENTS:
        s = d[(d.patientid == p) & d[sig].notna()].copy()
        if len(s) < 100:
            continue
        s['q'] = pd.qcut(s[sig], 5, labels=False, duplicates='drop')
        mm = s.groupby('q').spatial_stemness.mean()
        if mm.std() > 0:
            profs.append((mm - mm.mean()) / mm.std())
    P = pd.concat(profs, axis=1).mean(axis=1)
    peak = int(P.values.argmax()) + 1
    shape = 'monotonic increase' if peak == 5 else (
            'monotonic decrease' if peak == 1 else f'PEAK AT Q{peak} (inverted-U)')
    print(f'{sig:14s} Q1..Q5 = ' + '  '.join(f'{v:+.2f}' for v in P.values) +
          f'   -> {shape}')
    q_rows.append({'signature': sig, **{f'Q{i+1}': round(v, 3) for i, v in enumerate(P.values)},
                   'peak_quintile': peak, 'shape': shape})

print('\n=== (b) QUADRATIC TERM (within patient, standardised EMT) ===')
print('    negative & significant b2 = inverted-U\n')
quad_rows = []
for sig in EMT_SIGS + ['EMT_tan_epi']:
    n_neg = n_pos = 0
    details = []
    for p in PATIENTS:
        s = d[(d.patientid == p) & d[sig].notna()]
        if len(s) < 100:
            continue
        x = (s[sig] - s[sig].mean()) / s[sig].std()
        X = sm.add_constant(np.column_stack([x, x**2]))
        fit = sm.OLS(s.spatial_stemness.values, X).fit()
        b2, t2 = fit.params[2], fit.tvalues[2]
        if abs(t2) > 2:
            if b2 < 0: n_neg += 1
            else:      n_pos += 1
        details.append(f'{p[:8]}:{b2:+.3f}(t={t2:+.1f})')
    print(f'{sig:14s} significant NEGATIVE b2: {n_neg}/6  |  POSITIVE b2: {n_pos}/6')
    print(f'{"":14s} {"  ".join(details)}')
    quad_rows.append({'signature': sig, 'n_negative_quadratic': n_neg,
                      'n_positive_quadratic': n_pos,
                      'inverted_U_support': 'yes' if n_neg >= 4 else 'no'})

print('\n=== (c) LOWESS SHAPE CHECK (pooled within-patient z-scores) ===\n')
from statsmodels.nonparametric.smoothers_lowess import lowess
low_rows = []
for sig in ['pEMT_puram', 'EMT_tan_epi']:
    zs = []
    for p in PATIENTS:
        s = d[(d.patientid == p) & d[sig].notna()].copy()
        if len(s) < 100:
            continue
        s['zx'] = (s[sig] - s[sig].mean()) / s[sig].std()
        s['zy'] = (s.spatial_stemness - s.spatial_stemness.mean()) / s.spatial_stemness.std()
        zs.append(s[['zx', 'zy']])
    Z = pd.concat(zs)
    fit = lowess(Z.zy, Z.zx, frac=0.4, return_sorted=True)
    grid = np.linspace(-2, 2, 9)
    vals = np.interp(grid, fit[:, 0], fit[:, 1])
    print(f'{sig:14s} smoothed stemness at EMT z = ' +
          '  '.join(f'{v:+.2f}' for v in vals))
    print(f'{"":14s} (x = ' + '  '.join(f'{g:+.1f}' for g in grid) + ')')
    low_rows.append({'signature': sig, **{f'z{g:+.1f}': round(v, 3) for g, v in zip(grid, vals)}})

print('\n\n' + '=' * 72)
print('STEP 4 SUMMARY')
print('=' * 72)
summary = pd.DataFrame([
    {'test': 'refined epithelial-specific signature',
     'result': 'correlation collapses (0.196 -> 0.018) but score no longer measures EMT',
     'verdict': 'inconclusive - filter removes signal, not noise'},
    {'test': 'carried epithelial EMT (CytoSPACE)',
     'result': 'rho=0.513 but shuffled null = 0.463',
     'verdict': 'inconclusive - 90% structural'},
    {'test': 'tumor purity gradient',
     'result': 'Puram 0.196 -> 0.069 (>=50%) -> 0.025 (>=60%)',
     'verdict': 'association weakens strongly with purity'},
    {'test': 'leave-one-patient-out',
     'result': 'all 6 leave-outs keep 5/5 positive, shift < 0.10',
     'verdict': 'robust - not driven by one patient'},
    {'test': 'front definition thresholds (9 variants)',
     'result': "Cohen's d 0.216-0.239, 3/4 in every variant",
     'verdict': 'null result is not a threshold artefact'},
    {'test': 'Tan mesenchymal arm',
     'result': 'rho vs CAF = +0.775',
     'verdict': 'measures stromal content'},
    {'test': 'Tan epithelial arm',
     'result': 'rho vs stemness = -0.290 (0/6 positive), CAF-independent',
     'verdict': 'cleanest signal - loss of epithelial identity tracks stemness'},
])
print(summary.to_string(index=False))

pd.DataFrame(q_rows).to_csv(f'{RESULTS}/invertedU_quintiles.csv', index=False)
pd.DataFrame(quad_rows).to_csv(f'{RESULTS}/invertedU_quadratic.csv', index=False)
pd.DataFrame(low_rows).to_csv(f'{RESULTS}/invertedU_lowess.csv', index=False)
summary.to_csv(f'{RESULTS}/step4_summary.csv', index=False)
print(f'\nSaved -> invertedU_quintiles.csv, invertedU_quadratic.csv, '
      f'invertedU_lowess.csv, step4_summary.csv')

=== (a) QUINTILE PROFILE: mean stemness across EMT bins ===

pEMT_puram     Q1..Q5 = -0.63  -0.69  -0.46  +0.09  +1.69   -> monotonic increase
EMT_hallmark   Q1..Q5 = -0.75  -0.64  -0.45  +0.21  +1.62   -> monotonic increase
EMT_tan        Q1..Q5 = -1.05  -0.63  -0.19  +0.38  +1.49   -> monotonic increase
EMT_tan_epi    Q1..Q5 = +1.39  +0.36  -0.01  -0.53  -1.21   -> monotonic decrease

=== (b) QUADRATIC TERM (within patient, standardised EMT) ===
    negative & significant b2 = inverted-U

pEMT_puram     significant NEGATIVE b2: 0/6  |  POSITIVE b2: 5/6
               1142243F:+0.006(t=+6.5)  1160920F:+0.008(t=+9.0)  CID4290:+0.013(t=+9.7)  CID4465:+0.021(t=+5.3)  CID44971:+0.008(t=+4.5)  CID4535:-0.003(t=-2.0)
EMT_hallmark   significant NEGATIVE b2: 1/6  |  POSITIVE b2: 5/6
               1142243F:+0.008(t=+7.9)  1160920F:+0.006(t=+7.7)  CID4290:+0.008(t=+6.0)  CID4465:+0.021(t=+5.3)  CID44971:+0.006(t=+3.0)  CID4535:-0.007(t=-4.8)
EMT_tan        significant NEGATIVE b2: 1/6  |  POSI